# Trabalho 5 – Regressão SVM com PCA

## Importação das bibliotecas

In [7]:
import pandas as pd
import numpy as np

from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

## Preparando a base de dados

In [8]:
StudantPerformance_df = pd.read_csv('Student_Performance.csv')

StudantPerformance_df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


### Convertendo a variável categórica (Extracurricular Activities) para 0 e 1

0 -> Não \
1 -> Sim

In [9]:
col = 'Extracurricular Activities'
if StudantPerformance_df[col].dropna().isin(['Yes', 'No']).all():
    StudantPerformance_df[col] = StudantPerformance_df[col].map({'Yes': 1, 'No': 0})

StudantPerformance_df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0


Verificando colunas com valores nulos

In [10]:
print(StudantPerformance_df.isnull().sum())

Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64


Pegando apenas uma pequena amostra dos dados para o teste inicial do algorítmo

In [11]:
sample_df = StudantPerformance_df.sample(n=50, random_state=42).reset_index(drop=True)
sample_df.shape

(50, 6)

### Otimizando hiperparâmetros

In [12]:
# Separando X e y
X = sample_df.drop('Performance Index', axis=1).values
y = sample_df['Performance Index'].values

# Normalizando X (apenas os atributos de entrada)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Definindo os hiperparâmetros para o grid
kernels = ['linear', 'rbf', 'poly']
Cs = [1e-2, 1e-1, 1, 10]
gammas = [1e-2, 1e-1, 1, 10,100]

# K-Fold com K = 10
kf = KFold(n_splits=10, shuffle=True, random_state=42)

results = []

# Grid Search
for kernel in kernels:
    for C in Cs:
        for gamma in gammas:
            if kernel == 'linear':
                model = SVR(kernel=kernel, C=C)
            elif kernel == 'poly':
                model = SVR(kernel=kernel, C=C, gamma=gamma, degree=2)
            else:  # RBF
                model = SVR(kernel=kernel, C=C, gamma=gamma)

            mse_scores = []
            for train_index, test_index in kf.split(X_scaled):
                X_train, X_test = X_scaled[train_index], X_scaled[test_index]
                y_train, y_test = y[train_index], y[test_index]

                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                mse = mean_squared_error(y_test, y_pred)
                mse_scores.append(mse)

            avg_mse = np.mean(mse_scores)
            results.append({
                'kernel': kernel,
                'C': C,
                'gamma': gamma if kernel != 'linear' else 'N/A',
                'avg_mse': avg_mse
            })

# Ordenar pelo menor erro médio
results = sorted(results, key=lambda x: x['avg_mse'])

# Mostrar os melhores
print("Top 5 combinações com menor MSE:")
for r in results[:5]:
    print(r)


Top 5 combinações com menor MSE:
{'kernel': 'linear', 'C': 10, 'gamma': 'N/A', 'avg_mse': np.float64(8.010438124308472)}
{'kernel': 'linear', 'C': 10, 'gamma': 'N/A', 'avg_mse': np.float64(8.010438124308472)}
{'kernel': 'linear', 'C': 10, 'gamma': 'N/A', 'avg_mse': np.float64(8.010438124308472)}
{'kernel': 'linear', 'C': 10, 'gamma': 'N/A', 'avg_mse': np.float64(8.010438124308472)}
{'kernel': 'linear', 'C': 10, 'gamma': 'N/A', 'avg_mse': np.float64(8.010438124308472)}
